In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


from xgboost import XGBRegressor
from pymongo import MongoClient
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline


In [2]:
# ==================================================
# LOAD PROCESSED DATASET
# ==================================================

import pandas as pd

df = pd.read_csv("processed_dataset.csv")

print(f"✅ Archivo cargado correctamente: {df.shape[0]} filas y {df.shape[1]} columnas")
print("\nPrimeras filas:")
df.head()

✅ Archivo cargado correctamente: 66954 filas y 26 columnas

Primeras filas:


,RACEID,DRIVER_POINTS_BEFORE_RACE,POINTS,DRIVERID,DRIVERREF,LAPS,MILLISECONDS,WEATHER_rain,WEATHER_WET,SCORE,...,PS_COUNT,SC_COUNT,YEAR,ROLLING_POINTS,ROLLING_LAP,LAP_CONSISTENCY,RACE_INTERRUPTIONS,OVERTAKE_RATIO,DRIVER_ENCODED,RACE_ENCODED
0,833,NaN,-0.375795,579,fangio,62.0,7.105023e+06,NaN,NaN,5.769466,...,NaN,NaN,1950.0,-0.375795,NaN,NaN,NaN,0.0,241,1
1,833,NaN,-0.375795,589,chiron,26.0,2.979526e+06,NaN,NaN,5.769466,...,NaN,NaN,1950.0,-0.375795,NaN,NaN,NaN,0.0,159,1
2,833,NaN,-0.375795,619,gerard,67.0,7.678009e+06,NaN,NaN,5.769466,...,NaN,NaN,1950.0,-0.375795,NaN,NaN,NaN,0.0,295,1
3,833,NaN,-0.316728,627,rosier,68.0,7.792606e+06,NaN,NaN,5.769466,...,NaN,NaN,1950.0,-0.316728,NaN,NaN,NaN,0.0,678,1
4,833,NaN,-0.375795,640,graffenried,36.0,4.125497e+06,NaN,NaN,5.769466,...,NaN,NaN,1950.0,-0.375795,NaN,NaN,NaN,0.0,314,1


In [3]:
## SVM

In [4]:
# ============================================
# SVM REGRESSION - PREDICTING SCORE
# ============================================

# ============================================
# FEATURES & TARGET
# ============================================

features = [
    "DRIVER_POINTS_BEFORE_RACE",
    "LAPS",
    "MILLISECONDS",
    "WEATHER_rain",
    "WEATHER_WET",
    "WEATHER_cloudy",
    "OVERTAKEN_POSITIONS_TOTAL",
    "DNF_COUNT",
    "LAPMEAN",
    "FASTESTLAP",
    "PS_COUNT",
    "SC_COUNT",
    "ROLLING_POINTS",
    "ROLLING_LAP",
    "LAP_CONSISTENCY",
    "RACE_INTERRUPTIONS",
    "OVERTAKE_RATIO",
    "DRIVER_ENCODED",
    "RACE_ENCODED"
]

target = "SCORE"

# ============================================
# CREATE CLEAN DATAFRAME
# ============================================

model_df = df[features + [target]].copy()

# replace inf
model_df = model_df.replace([np.inf, -np.inf], np.nan)

# remove missing rows
model_df = model_df.dropna()

# ============================================
# X AND y
# ============================================

X = model_df[features]

# make y 1D
y = model_df[target].values.ravel()



# ============================================
# TRAIN / TEST SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ============================================
# SVM PIPELINE
# ============================================

# IMPORTANT:
# SVM is VERY sensitive to scaling,
# so StandardScaler is essential.

svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(
        kernel="rbf",   # most common kernel
        C=100,
        gamma="scale",
        epsilon=0.1
    ))
])

# ============================================
# TRAIN MODEL
# ============================================

svm_model.fit(X_train, y_train)

# ============================================
# PREDICTIONS
# ============================================

y_pred = svm_model.predict(X_test)

# ============================================
# EVALUATION
# ============================================

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("========== SVM RESULTS ==========")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2   : {r2:.4f}")

# ============================================
# SAMPLE PREDICTIONS
# ============================================

results = pd.DataFrame({
    "Actual_SCORE": y_test,
    "Predicted_SCORE": y_pred
})

print("\nSample predictions:")
print(results.head(10))

# ============================================
# OPTIONAL: SAVE MODEL
# ============================================

import joblib

joblib.dump(svm_model, "svm_score_model.pkl")

print("\nModel saved as svm_score_model.pkl")

========== SVM RESULTS ==========
MAE  : 0.2737
RMSE : 0.4524
R2   : 0.8548

Sample predictions:
   Actual_SCORE  Predicted_SCORE
0         5.440         5.600820
1         5.400         6.009885
2         7.040         6.907063
3         5.440         5.598221
4         6.370         6.166034
5         6.300         6.124041
6         5.998         6.016134
7         6.770         6.607508
8         6.480         6.380023
9         8.760         7.508064

Model saved as svm_score_model.pkl


In [5]:
## XGBOOST

In [6]:
# ============================================
# XGBOOST REGRESSION - PREDICTING SCORE
# ============================================

# ============================================
# CLEAN TARGET
# ============================================
# ============================================
# CLEAN DATA FOR XGBOOST
# ============================================

# Crear dataframe solo con features y target
model_df = df[features + [target]].copy()

# Convertir TODO a numérico
for col in model_df.columns:
    model_df[col] = pd.to_numeric(
        model_df[col],
        errors="coerce"
    )

# Reemplazar infinitos
model_df = model_df.replace(
    [np.inf, -np.inf],
    np.nan
)

# Ver cuántos NaNs hay
print(model_df.isna().sum())

# Eliminar filas problemáticas
model_df = model_df.dropna()


# ============================================
# X AND y
# ============================================

X = model_df[features]
y = model_df[target]


# ============================================
# TRAIN / TEST SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ============================================
# XGBOOST MODEL
# ============================================

xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

# ============================================
# TRAIN MODEL
# ============================================

xgb_model.fit(X_train, y_train)

# ============================================
# PREDICTIONS
# ============================================

y_pred = xgb_model.predict(X_test)

# ============================================
# EVALUATION
# ============================================

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("========== XGBOOST RESULTS ==========")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2   : {r2:.4f}")

# ============================================
# FEATURE IMPORTANCE
# ============================================

importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": xgb_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print("\nTop Feature Importances:")
print(importance_df)

# ============================================
# SAMPLE PREDICTIONS
# ============================================

results = pd.DataFrame({
    "Actual_SCORE": y_test.values,
    "Predicted_SCORE": y_pred
})

print("\nSample predictions:")
print(results.head(10))

# ============================================
# OPTIONAL: SAVE MODEL
# ============================================

import joblib

joblib.dump(xgb_model, "xgboost_score_model.pkl")

print("\nModel saved as xgboost_score_model.pkl")

DRIVER_POINTS_BEFORE_RACE    33477
LAPS                         17202
MILLISECONDS                 17202
WEATHER_rain                 33827
WEATHER_WET                  33827
WEATHER_cloudy                 700
OVERTAKEN_POSITIONS_TOTAL      700
DNF_COUNT                      700
LAPMEAN                      60725
FASTESTLAP                   61379
PS_COUNT                     58376
SC_COUNT                     61182
ROLLING_POINTS                   0
ROLLING_LAP                  59977
LAP_CONSISTENCY              62555
RACE_INTERRUPTIONS           62488
OVERTAKE_RATIO               22616
DRIVER_ENCODED                   0
RACE_ENCODED                     0
SCORE                          700
dtype: int64
========== XGBOOST RESULTS ==========
MAE  : 0.0594
RMSE : 0.1245
R2   : 0.9890

Top Feature Importances:
                      Feature  Importance
7                   DNF_COUNT    0.248398
16             OVERTAKE_RATIO    0.224292
6   OVERTAKEN_POSITIONS_TOTAL    0.159727
9            